# Qwen3 Sycophancy Baseline — Milestone 1

실제 GPU 실행용 인터페이스입니다. 연구 로직은 저장소의 `src/truthgate`와 CLI에 있으며 이 노트북에서는 재구현하지 않습니다.

**실행 전:** Colab에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. 기본 저장소는 [seungjoolee24/XAI_research](https://github.com/seungjoolee24/XAI_research)입니다. 준비한 저장소 경로를 사용할 때에는 `EXISTING_REPO_DIR`로 지정하세요. 이 파일에는 미리 실행한 출력이 없습니다. 실제 Qwen3 GPU 실행 이후에 표시되는 값만 모델 결과로 사용하세요.

모든 연구 추론은 **사전 PDF 생성·표시·다운로드 → 다음 셀에서 GPU 실행 → 성공·실패 사후 PDF 표시·다운로드** 순서입니다. 사전 PDF에는 목적, 정확한 데이터·조건, 핵심 의사코드, 유효 설정과 계획 지표가 포함됩니다. 사후 PDF에는 실제 상태, 통계와 분모, invalid 응답, flip 유형, 관찰과 해석 한계가 포함됩니다. `DOWNLOAD_REPORTS=True`를 유지하고 브라우저에서 파일 다운로드를 허용하세요. 설치와 offline 단위 테스트는 소프트웨어 검증이며 연구 실험이 아닙니다.

범위: 단일 추론, neutral / false_opinion / correct_opinion, 후보 점수, hidden-state 추출. Training, probing, logit lens, intervention은 포함하지 않습니다.

Colab에서 파일 → 노트북 열기 → GitHub로 열거나 이 노트북을 업로드할 수 있습니다. GitHub 없이 실행하려면 [README](../README.md)의 소스 ZIP 업로드 절차 후 `EXISTING_REPO_DIR`을 지정하세요. 토큰을 코드에 저장하지 않습니다.


In [ ]:
from pathlib import Path

# 사용자 GitHub 저장소에서 clone하거나 이미 준비한 경로를 지정합니다.
REPO_URL = "https://github.com/seungjoolee24/XAI_research"
EXISTING_REPO_DIR = ""  # 예: "/content/latent-truth-xai" (업로드 후 압축 해제한 폴더)
REPO_REF = ""  # 재현할 코드 commit이 있으면 입력; 비워두면 clone한 기본 branch
CLONE_DIR = Path("/content/latent-truth-xai")
INCLUDE_SWAPPED = False  # True: 선택지 교환본까지 포함 (60행 -> 120행)
BASELINE_LIMIT = None  # None: 전체 20문항; 빠른 부분 실행은 1 등 양의 정수
HIDDEN_LIMIT = 3  # 1~3개 원본 질문. INCLUDE_SWAPPED는 hidden 비교에도 적용
DOWNLOAD_REPORTS = True  # 모든 연구 실행의 사전·사후 PDF를 브라우저로 전달


## 1. GPU 확인

CUDA가 없으면 중단합니다. GPU 실험을 CPU로 조용히 전환하지 않습니다. 모델은 양자화 없이 단일 GPU에 올라가므로 충분한 GPU 메모리가 필요합니다.


In [ ]:
import sys
import subprocess
import torch

assert torch.cuda.is_available(), "GPU runtime is required. Select a Colab GPU runtime and reconnect."
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("Automatic dtype:", "bfloat16" if torch.cuda.is_bf16_supported(including_emulation=False) else "float16")


## 2. 저장소와 의존성

GitHub에서 clone하거나 이미 준비된 폴더를 사용합니다. 기존 폴더를 자동 초기화하거나 덮어쓰지 않습니다. 설치된 CUDA PyTorch가 `torch>=2.6,<3` 조건을 만족하면 유지합니다. CPU 전용 PyTorch index를 사용하지 않습니다.


In [ ]:
if EXISTING_REPO_DIR.strip():
    REPO_DIR = Path(EXISTING_REPO_DIR).expanduser().resolve()
    if not REPO_DIR.is_dir():
        raise FileNotFoundError(f"Repository directory does not exist: {REPO_DIR}")
else:
    if not REPO_URL.strip() or REPO_URL == "YOUR_GITHUB_REPO":
        raise ValueError("Set REPO_URL to your actual GitHub repository URL, or set EXISTING_REPO_DIR.")
    if CLONE_DIR.exists():
        raise FileExistsError(f"{CLONE_DIR} exists. Set EXISTING_REPO_DIR to reuse it explicitly.")
    subprocess.run(["git", "clone", "--", REPO_URL, str(CLONE_DIR)], check=True)
    REPO_DIR = CLONE_DIR.resolve()

if not (REPO_DIR / "pyproject.toml").is_file():
    raise FileNotFoundError(f"pyproject.toml is missing from {REPO_DIR}")
if REPO_REF.strip():
    subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR, check=True)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True)
else:
    print("Uploaded directory has no Git metadata; retain the source version separately.")
print("Repository:", REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=REPO_DIR, check=True)


Colab가 설치 후 런타임 재시작을 요구하면 재시작한 다음 설정 셀부터 다시 실행하세요. 이미 clone한 폴더는 `EXISTING_REPO_DIR = "/content/latent-truth-xai"`로 지정합니다. CLI는 새 Python 프로세스에서 실행하므로 설치된 패키지를 사용합니다.


In [ ]:
# CUDA 확인은 CLI와 같은 새 Python 프로세스에서도 수행합니다.
subprocess.run([
    sys.executable, "-c",
    "import torch, transformers; "
    "assert torch.cuda.is_available(), 'CUDA unavailable after installation'; "
    "print('torch', torch.__version__, 'transformers', transformers.__version__); "
    "print('GPU', torch.cuda.get_device_name(0))"
], cwd=REPO_DIR, check=True)

# 작은 offline fixtures / randomly initialized tiny Qwen3 검증: 과학적 모델 결과 아님
subprocess.run([sys.executable, "-m", "pytest"], cwd=REPO_DIR, check=True)


## 3. 설정과 결과 경로

기본 모델은 `Qwen/Qwen3-4B-Instruct-2507`이며 YAML의 commit SHA로 model/tokenizer를 고정합니다. 최대 입력 512 token, 생성 8 token, batch 1, greedy입니다. 입력 초과 시 오류로 중단하며 truncation하지 않습니다. BF16 지원 GPU는 BF16, 그 외 GPU는 FP16을 사용합니다.

각 실행 디렉터리는 새로 생성됩니다. 이미 존재하는 경로에는 쓰지 않습니다. 셀을 재실행하려면 아래 경로 셀을 다시 실행해 새 세션 경로를 받으세요.


In [ ]:
from datetime import datetime, timezone
import base64
import hashlib
import html
import json
import yaml
from IPython.display import HTML, IFrame, display

CONFIG_PATH = REPO_DIR / "configs" / "baseline.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_ROOT = REPO_DIR / "outputs" / f"colab_{run_stamp}"
SMOKE_DIR = RUN_ROOT / "smoke"
BASELINE_DIR = RUN_ROOT / "baseline"
HIDDEN_DIR = RUN_ROOT / "hidden"
print("Session output:", RUN_ROOT)


def show_pdf(pdf_path):
    """Supply the document before any following research inference cell runs."""
    pdf_path = Path(pdf_path)
    if not pdf_path.is_file():
        raise FileNotFoundError(f"Required experiment report is missing: {pdf_path}")
    data_url = "data:application/pdf;base64," + base64.b64encode(pdf_path.read_bytes()).decode("ascii")
    display(HTML(
        f'<p><a href="{data_url}" download="{html.escape(pdf_path.name, quote=True)}">'
        f'{html.escape(pdf_path.name)} 다운로드</a></p>'
    ))
    display(IFrame(src=data_url, width="100%", height=640))
    print("PDF:", pdf_path)
    if DOWNLOAD_REPORTS:
        from google.colab import files
        files.download(str(pdf_path))


def prepare_experiment(kind, script, output_dir, extra_args=()):
    """Generate and deliver a pre PDF without loading the model or creating its run directory."""
    if output_dir.exists():
        raise FileExistsError(f"Refusing to overwrite {output_dir}; rerun the path cell for a new session.")
    pre_dir = RUN_ROOT / "pre_reports" / kind
    if pre_dir.exists():
        raise FileExistsError(f"Pre report already exists at {pre_dir}; rerun the path cell for a new session.")
    extra_args = tuple(extra_args)
    command = [
        sys.executable, str(REPO_DIR / "scripts" / "generate_experiment_report.py"),
        "--config", str(CONFIG_PATH), "--kind", kind, "--phase", "pre",
        "--device", "cuda", "--output-dir", str(pre_dir), *extra_args,
    ]
    subprocess.run(command, cwd=REPO_DIR, check=True)
    pre_pdf = pre_dir / "pre_experiment.pdf"
    show_pdf(pre_pdf)
    return {
        "kind": kind, "script": script, "output_dir": output_dir,
        "extra_args": extra_args, "pre_pdf": pre_pdf,
        "config_sha256": hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest(),
        "pre_report_delivered": True,
    }


def run_experiment(plan):
    """Execute only a previously delivered plan, then deliver its actual post report even on failure."""
    if not plan.get("pre_report_delivered") or not plan["pre_pdf"].is_file():
        raise RuntimeError("Run the preceding pre-experiment PDF cell first.")
    if hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest() != plan["config_sha256"]:
        raise RuntimeError("Configuration changed after the pre PDF; prepare a new report in a new session.")
    output_dir = plan["output_dir"]
    if output_dir.exists():
        raise FileExistsError(f"Refusing to overwrite {output_dir}; prepare a new session.")
    command = [
        sys.executable, str(REPO_DIR / "scripts" / plan["script"]),
        "--config", str(CONFIG_PATH), "--device", "cuda",
        "--output-dir", str(output_dir), *plan["extra_args"],
    ]
    print("Command arguments:", command)
    try:
        subprocess.run(command, cwd=REPO_DIR, check=True)
    finally:
        post_pdf = output_dir / "post_experiment.pdf"
        if post_pdf.is_file():
            show_pdf(post_pdf)
        else:
            print("Post PDF was not produced. The process may have stopped before report initialization.")
            print("Retain all files and the traceback; do not report an incomplete run as completed.")


## 4. 단일 프롬프트 GPU smoke

모델 로드, chat template, 동일 prefix의 후보 점수, 자유 greedy generation을 확인합니다. `predicted_answer`는 전체 생성 문자열이 정확히 A 또는 B일 때만 유효합니다. 설명 문장 등은 invalid로 보존하며 candidate winner로 대체하지 않습니다.

아래 사전 PDF를 먼저 확인한 다음 **다음 GPU 실행 셀**을 실행합니다. 이 smoke도 실제 연구 추론이므로 사전·사후 보고서를 각각 제공합니다.


In [ ]:
smoke_plan = prepare_experiment("smoke", "smoke_inference.py", SMOKE_DIR)
print("사전 PDF를 전달했습니다. 다음 셀에서 단일 프롬프트 GPU 추론을 실행합니다.")


In [ ]:
run_experiment(smoke_plan)
print("Smoke outputs:", sorted(path.name for path in SMOKE_DIR.iterdir()))


## 5. 전체 baseline

원문 20문항은 정답 A 10개 / B 10개입니다. 세 조건을 모두 실행하면 60행입니다. `INCLUDE_SWAPPED=True`는 교환본을 추가해 120행을 만듭니다. 같은 질문의 교환본은 독립적인 사실 질문이 아닙니다. 실제 생성 답변 flip과 후보 winner 변화는 따로 저장합니다.

한 문항만 먼저 실행하려면 설정 셀의 `BASELINE_LIMIT=1`을 지정하세요. 사전 보고서와 실제 실행이 같은 부분집합과 설정을 사용합니다. 설정·limit·조건을 바꿀 때에는 경로 셀부터 다시 실행해 새 세션을 준비하세요.


In [ ]:
baseline_args = ["--include-swapped"] if INCLUDE_SWAPPED else []
if BASELINE_LIMIT is not None:
    if not isinstance(BASELINE_LIMIT, int) or isinstance(BASELINE_LIMIT, bool) or BASELINE_LIMIT < 1:
        raise ValueError("BASELINE_LIMIT must be None or a positive integer.")
    baseline_args += ["--limit", str(BASELINE_LIMIT)]
baseline_plan = prepare_experiment("baseline", "run_baseline.py", BASELINE_DIR, baseline_args)
print("사전 PDF를 전달했습니다. 다음 셀에서 baseline GPU 추론을 실행합니다.")


In [ ]:
run_experiment(baseline_plan)


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary = pd.read_csv(BASELINE_DIR / "summary.csv")
results = pd.read_csv(BASELINE_DIR / "results.csv")
flip_cases = pd.read_csv(BASELINE_DIR / "flip_cases.csv")
print("Condition summary (actual Qwen3 GPU run):")
display(summary)

result_columns = [
    "question_id", "option_order", "ground_truth", "condition", "generated_text",
    "predicted_answer", "candidate_answer", "is_correct", "score_type",
    "logit_a", "logit_b", "logit_margin", "score_margin", "probability_a", "probability_b",
    "answer_flip_from_neutral", "correct_to_incorrect", "incorrect_to_correct",
]
display(results[result_columns])

print("Actual generated-answer flip cases:")
if flip_cases.empty:
    print("No generated-answer flips in this run.")
else:
    display(flip_cases)

candidate_only = results[
    results["candidate_flip_from_neutral"].eq(True)
    & results["answer_flip_from_neutral"].eq(False)
]
print("Candidate-only changes:")
if candidate_only.empty:
    print("No candidate-only changes with an unchanged valid generated answer.")
else:
    display(candidate_only[result_columns + ["candidate_flip_from_neutral"]])


후보 둘에 정규화한 `probability_a/b`는 calibrated factual confidence가 아닙니다. `score_type`이 단일 token이면 raw next-token logit, 다중 token이면 합산 sequence conditional log probability를 비교합니다. `candidate_logprob_a/b`도 확인할 수 있습니다. 다중 token 후보의 raw logit과 raw logit margin은 `null`입니다.

Flip은 `(question_id, option_order)`로 짝지으며 invalid pair는 방향성 flip으로 세지 않습니다. 정확도에는 invalid가 오답으로 포함됩니다. `summary.json`의 `metric_definitions`에서 비율별 분모를 확인하세요. Correct opinion은 정답 방향 의견 대조군이며 증거 기반 rational updating 실험은 아닙니다.


In [ ]:
summary_detail = json.loads((BASELINE_DIR / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary_detail.get("metric_definitions", {}), indent=2, ensure_ascii=False))
metadata = json.loads((BASELINE_DIR / "run_metadata.json").read_text(encoding="utf-8"))
print("Retain this metadata with the results:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))


## 6. Hidden-state API 확인

세 질문에서 neutral 대비 두 opinion 조건을 비교합니다. 기본 Qwen3 모델은 36 block, hidden size 2560이므로 반환 state는 37개입니다: index 0 embedding, 1–35 중간 block 출력, 36 final normalization 적용 후 출력입니다. shape는 `[1, T, 2560]`입니다.

비교 위치는 각 rendered prompt의 마지막 token으로, 보통 assistant generation suffix에 속합니다. 같은 질문 속 token을 정렬한 것이 아니며 길이가 달라 절대 위치도 다를 수 있습니다. shape, norm, cosine, 위치만 파일에 저장합니다. 작은 last-position 벡터만 일시 보관하고 전체 activation은 저장하지 않습니다.


In [ ]:
if not isinstance(HIDDEN_LIMIT, int) or isinstance(HIDDEN_LIMIT, bool) or not 1 <= HIDDEN_LIMIT <= 3:
    raise ValueError("HIDDEN_LIMIT must be an integer from 1 to 3.")
hidden_args = ["--limit", str(HIDDEN_LIMIT)]
if INCLUDE_SWAPPED:
    hidden_args += ["--include-swapped"]
hidden_plan = prepare_experiment("hidden", "inspect_hidden_states.py", HIDDEN_DIR, hidden_args)
print("사전 PDF를 전달했습니다. 다음 셀에서 hidden-state GPU 추론을 실행합니다.")


In [ ]:
run_experiment(hidden_plan)
hidden = pd.read_csv(HIDDEN_DIR / "hidden_states.csv")
display(hidden.head(15))
print("Observed returned-state counts per question/order/condition:")
display(hidden.groupby(["question_id", "option_order", "condition"])["state_index"].nunique().rename("state_count").reset_index())
from IPython.display import Image

display(Image(filename=str(HIDDEN_DIR / "hidden_comparison.png")))


Cosine과 norm은 API가 정상 동작하는지 확인하는 진단입니다. 이 값만으로 latent truth, 지식 해독, 최종 답변에 대한 인과적 사용을 입증할 수 없습니다. H1–H3는 이번 단계에서 검증하지 않습니다. 실제 모델 결과를 보고할 때에는 offline 테스트와 구분하고 모델 revision, 데이터 digest, seed, GPU/dtype, dependency 버전과 유효 설정을 함께 보관하세요.


## 7. 선택: 결과 ZIP 다운로드

아래 셀은 자동으로 외부에 업로드하거나 Google Drive를 mount하지 않습니다. ZIP을 원하면 `DOWNLOAD_RESULTS=True`로 바꾸고 실행하세요. ZIP에는 **모든 사전·사후 PDF**, 유효 설정, 재현성 metadata와 이번 세션의 작은 실행 결과를 포함합니다. 가중치와 모델 cache는 포함하지 않습니다. PDF 개별 전달은 위 `DOWNLOAD_REPORTS=True` 설정으로 각 실험 전후 수행됩니다.


In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    if not RUN_ROOT.is_dir():
        raise FileNotFoundError("Run at least one experiment before downloading results.")
    archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT.parent, base_dir=RUN_ROOT.name)
    print("Result archive:", archive)
    files.download(archive)
else:
    print("Download skipped. Results are available at:", RUN_ROOT)


공식 참고: [Qwen3-4B-Instruct-2507 모델 카드](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507), [Transformers chat templates](https://huggingface.co/docs/transformers/v4.57.1/en/chat_templating), [Qwen3 API](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/qwen3). 설치 pin과 문서 버전은 구분하며 실제 실행 버전은 metadata를 확인하세요.
